In [ ]:
import datetime as dt
import polars as pl
from trading_engine.core import (
    read_data, create_model_state, orchestrate_model_backtests,
    orchestrate_model_simulations, orchestrate_portfolio_backtests,
    orchestrate_portfolio_simulations
)
from trading_engine.models import MODELS
from trading_engine.optimizers import OPTIMIZERS

from trading_engine.model_state.registry import momentum
from common.constants import ProcessingMode

import datetime

In [ ]:
# 1) experiment config
universe = [
  "SPY-US", "SLV-US", "GLD-US", "TLT-US", "USO-US", "UNG-US", "IXJ-US",
  "KXI-US", "JXI-US", "IXG-US", "IXN-US", "RXI-US", "MXI-US", "EXI-US",
  "IXC-US", "IEI-US", "SHY-US", "BIL-US", "JPXN-US", "INDA-US", "MCHI-US",
  "EZU-US", "IBIT-US", "ETHA-US", "VIXY-US"
]
features = ["close_momentum_10"]                 # must exist in FEATURES
models   = ["RXI_TLT_pml_10", "GLD_USO_nml_10"]    # keys in MODELS
optimizers   = ["equal_weight"]                    # keys in OPTIMIZERS
initial_value = 1_000_000
start_date = datetime.date(2021, 1, 1)
end_date = datetime.date(2025, 1, 1)

In [ ]:
# 2) build model state + prices (cached locally)
lf = read_data()
model_state, prices = create_model_state(
    lf=lf,
    features=features,
    start_date=start_date,
    end_date=end_date,
    universe=universe
)

In [ ]:
# 3) run model backtests + simulations
model_insights = orchestrate_model_backtests(
    model_state=model_state,
    models=models,
    universe=universe
)

model_simulations = orchestrate_model_simulations(
    prices=prices,
    model_insights=model_insights
)

In [ ]:
# 4) optimize portfolio and simulate
portfolio_insights = orchestrate_portfolio_backtests(
    model_insights=model_insights,
    backtest_results=model_simulations,
    universe=universe,
    optimizers=optimizers
)

portfolio_simulations = orchestrate_portfolio_simulations(
    prices=prices,
    portfolio_insights=portfolio_insights
)

In [ ]:
# 5) visualize one result (example: equal_weight)
portfolio_simulations["equal_weight"]["backtest_metrics"]